# 01 — Dataset inventory

This notebook creates a first overview of the banking CSV files: row counts, columns, inferred data types, and missing-value counts.

Place the CSVs in data/raw/ and run the notebook from the project folder. The scan reads in chunks and prints metadata only; it does not display customer rows.

The data contains customer and financial details. Keep notebook outputs free of individual records before publishing.


In [1]:
from pathlib import Path
import pandas as pd

# Find the project folder whether the notebook starts in the root or notebooks/.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((folder for folder in candidates if (folder / 'data' / 'raw').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find data/raw. Open the project folder and place the CSV files in data/raw/.')

DATA_DIR = ROOT / 'data' / 'raw'
REPORTS_DIR = ROOT / 'reports'
files = sorted(DATA_DIR.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR}')

print(f'Found {len(files)} CSV files in {DATA_DIR}')


Found 7 CSV files in c:\Users\HP\OneDrive\Desktop\Banking dataset\Banking-data-analysis\data\raw


In [2]:
def profile_csv(path, chunksize=100_000):
    # Read only the header first, then scan the data a chunk at a time.
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    missing = pd.Series(0, index=columns, dtype='int64')
    rows = 0
    first_chunk_types = {}

    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        rows += len(chunk)
        missing = missing.add(chunk.isna().sum().astype('int64'), fill_value=0).astype('int64')
        if not first_chunk_types:
            first_chunk_types = {column: str(dtype) for column, dtype in chunk.dtypes.items()}

    return {
        'file': path.name,
        'rows': rows,
        'column_count': len(columns),
        'columns': columns,
        'inferred_types_first_chunk': first_chunk_types,
        'missing': missing[missing > 0].to_dict(),
    }


In [3]:
# This scans each CSV fully but keeps only one chunk in memory at a time.
profiles = [profile_csv(path) for path in files]

inventory = pd.DataFrame([
    {
        'file': item['file'],
        'rows': item['rows'],
        'columns': item['column_count'],
        'columns_with_missing_values': len(item['missing']),
        'missing_value_cells': sum(item['missing'].values()),
    }
    for item in profiles
]).sort_values('rows', ascending=False)

display(inventory)


,file,rows,columns,columns_with_missing_values,missing_value_cells
2,card_transactions.csv,3000000,6,0,0
6,loan_payments.csv,600000,7,0,0
0,accounts.csv,95000,7,0,0
3,cards.csv,65000,8,0,0
4,customers.csv,60000,12,0,0
5,employees.csv,1800,6,0,0
1,branches.csv,150,6,0,0


In [4]:
# Print each table's schema and only the fields with missing values.
for item in profiles:
    print(f"\n{item['file']} — {item['rows']:,} rows, {item['column_count']} columns")
    print('Columns:', ', '.join(item['columns']))
    print('Inferred types from first chunk:', item['inferred_types_first_chunk'])
    print('Missing value counts:', item['missing'] if item['missing'] else 'None found')



accounts.csv — 95,000 rows, 7 columns
Columns: account_id, customer_id, branch_id, account_type, balance, open_date, status
Inferred types from first chunk: {'account_id': 'int64', 'customer_id': 'int64', 'branch_id': 'int64', 'account_type': 'object', 'balance': 'float64', 'open_date': 'object', 'status': 'object'}
Missing value counts: None found

branches.csv — 150 rows, 6 columns
Columns: branch_id, branch_name, city, state, opened_date, ifsc_code
Inferred types from first chunk: {'branch_id': 'int64', 'branch_name': 'object', 'city': 'object', 'state': 'object', 'opened_date': 'object', 'ifsc_code': 'object'}
Missing value counts: None found

card_transactions.csv — 3,000,000 rows, 6 columns
Columns: card_txn_id, card_id, txn_date, merchant_category, amount, is_fraud
Inferred types from first chunk: {'card_txn_id': 'int64', 'card_id': 'int64', 'txn_date': 'object', 'merchant_category': 'object', 'amount': 'float64', 'is_fraud': 'int64'}
Missing value counts: None found

cards.csv

In [5]:
# Save a small metadata-only inventory for the reports folder.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
inventory_path = REPORTS_DIR / 'csv_inventory.csv'
inventory.to_csv(inventory_path, index=False)
print(f'Saved metadata summary to {inventory_path}')


Saved metadata summary to c:\Users\HP\OneDrive\Desktop\Banking dataset\Banking-data-analysis\reports\csv_inventory.csv


## Candidate table links

The column names suggest these relationships. This notebook does not yet prove that the IDs are unique or that every linked ID exists; check those in the loan analysis notebook.

- customers.customer_id → accounts.customer_id, cards.customer_id, loans.customer_id, support_tickets.customer_id
- accounts.account_id → transactions.account_id, cards.account_id
- branches.branch_id → accounts.branch_id, loans.branch_id, employees.branch_id
- cards.card_id → card_transactions.card_id
- loans.loan_id → loan_payments.loan_id

## What to learn from the output

- A row is one record, such as one loan payment or one card transaction.
- A column is a property of that record.
- Missing-value counts help you see which fields may need cleaning.
- Types are inferred from the first chunk. If a column mixes dates, numbers, and text, inspect it before analysis.
- The largest files may take time to scan. The chunked read limits memory use.
